In [ ]:
# ============================================================================
# Install Required Packages
# ============================================================================
!pip install thop -q
!pip install tqdm -q

print("✓ Packages installed successfully!")

# ============================================================================
# Import Libraries
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time
from tqdm import tqdm
import pandas as pd
from thop import profile, clever_format
import pickle
import os
import shutil
import warnings
from sklearn.exceptions import ConvergenceWarning

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

drive.mount('/content/drive')

DRIVE_RESULTS_FOLDER = '/content/drive/MyDrive/DLOps_Assignment_Results'
os.makedirs(DRIVE_RESULTS_FOLDER, exist_ok=True)
print(f"\n✓ Google Drive mounted")

def initialize_csv_files():

    q1a_headers = ['Dataset', 'Epochs', 'Batch Size', 'Optimizer', 'Learning Rate',
                   'Model', 'Test Accuracy (%)', 'Best Val Acc (%)', 'pin_memory']

    q1b_headers = ['Dataset', 'Kernel', 'Test Accuracy (%)', 'Train Time (ms)']

    q2_headers = ['Compute', 'Batch Size', 'Optimizer', 'Learning Rate', 'Model',
                  'Test Accuracy (%)', 'Train Time (ms)', 'FLOPs']

    for dataset in ['mnist','fashionmnist']:
        filename = f'{dataset}_q1a_results.csv'
        filepath = os.path.join(DRIVE_RESULTS_FOLDER, filename)
        pd.DataFrame(columns=q1a_headers).to_csv(filepath, index=False)
        print(f"✓ Created {filename}")

    for dataset in ['mnist', 'fashionmnist']:
        filename = f'{dataset}_q1b_svm_results.csv'
        filepath = os.path.join(DRIVE_RESULTS_FOLDER, filename)
        pd.DataFrame(columns=q1b_headers).to_csv(filepath, index=False)
        print(f"✓ Created {filename}")

    filename = 'fashionmnist_q2_cpu_gpu_results.csv'
    filepath = os.path.join(DRIVE_RESULTS_FOLDER, filename)
    pd.DataFrame(columns=q2_headers).to_csv(filepath, index=False)
    print(f"✓ Created {filename}")

print("\n📄 Initializing CSV files in KAGGLE...")
initialize_csv_files()

# ============================================================================
# Data Preparation Functions
# ============================================================================
def prepare_datasets(dataset_name='MNIST', batch_size=16, pin_memory=False):

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    if dataset_name == 'MNIST':
        mnist_train_dataset = datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform)
        mnist_test_dataset = datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform)
        mnist_combined = ConcatDataset([mnist_train_dataset, mnist_test_dataset])


    else:
        fashionmnist_train_dataset = datasets.FashionMNIST(root='./data', train=True,
                                             download=True, transform=transform)
        fashionmnist_test_dataset = datasets.FashionMNIST(root='./data', train=False,
                                             download=True, transform=transform)
        mnist_combined = ConcatDataset([fashionmnist_train_dataset, fashionmnist_test_dataset])

    total_size = len(mnist_combined)
    train_size = int(0.7 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        mnist_combined, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"Dataset: {dataset_name}")
    print(f"Train size: {len(train_dataset)}")
    print(f"Val size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                             shuffle=True, pin_memory=pin_memory, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                           shuffle=False, pin_memory=pin_memory, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            shuffle=False, pin_memory=pin_memory, num_workers=2)

    return train_loader, val_loader, test_loader

print("✓ Data preparation functions defined")

# ============================================================================
# Model Preparation Functions
# ============================================================================
def get_model(model_name='resnet18', num_classes=10):
    if model_name == 'resnet18':
        model = models.resnet18(weights=None)
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

print("✓ Model preparation functions defined")

# ============================================================================
# Training Functions
# ============================================================================
def train_epoch(model, train_loader, criterion, optimizer, device, use_amp=True):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    scaler = torch.amp.GradScaler('cuda') if use_amp and device.type == 'cuda' else None

    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        if use_amp and device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device, use_amp=True):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            if use_amp and device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def train_model(model, train_loader, val_loader, optimizer, epochs=10,
                device=device, use_amp=True):
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)

    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    epoch_times = []

    print(f"\n{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12} {'Time (s)':<10}")
    print("-" * 70)

    for epoch in range(epochs):
        epoch_start_time = time.time()

        train_loss, train_acc = train_epoch(model, train_loader, criterion,
                                           optimizer, device, use_amp)
        val_loss, val_acc = validate(model, val_loader, criterion, device, use_amp)

        epoch_time = time.time() - epoch_start_time

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time)

        print(f"{epoch+1:>2}/{epochs:<5} {train_loss:<12.4f} {train_acc:<12.2f} {val_loss:<12.4f} {val_acc:<12.2f} {epoch_time:<10.2f}")

    print("-" * 70)
    print(f"Average epoch time: {np.mean(epoch_times):.2f}s, Total time: {sum(epoch_times):.2f}s\n")

    return {
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'epoch_times': epoch_times
    }

def test_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    return accuracy

print("✓ Training functions defined")

# ============================================================================
# FLOPs Calculation Function
# ============================================================================
def count_flops(model, input_size=(1, 3, 224, 224)):
    model.eval()
    input_tensor = torch.randn(input_size)

    device = next(model.parameters()).device
    input_tensor = input_tensor.to(device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops_formatted, params_formatted = clever_format([flops, params], "%.3f")

    print(f"  FLOPs: {flops_formatted}, Params: {params_formatted}")

    return flops

print("✓ FLOPs calculation function defined")

# ============================================================================
# Q1(a) - ResNet Experiments Function
# ============================================================================
def run_q1a_experiments(dataset_name='MNIST'):

    results = []
    best_model_info = {'accuracy': 0, 'model': None, 'config': {}}

    batch_sizes = [16,32]
    optimizers_config = ['SGD', 'Adam']
    learning_rates = [0.001, 0.0001]
    models_config = ['resnet18', 'resnet50']
    pin_memory_options = [False, True]
    epochs_options = [3,5]
    csv_filepath = os.path.join(DRIVE_RESULTS_FOLDER, f'{dataset_name.lower()}_q1a_results.csv')

    print(f"\n{'='*80}")
    print(f"Running Q1(a) experiments on {dataset_name}")
    total_configs = (len(batch_sizes) * len(optimizers_config) * len(learning_rates) *
                    len(pin_memory_options) * len(models_config) * len(epochs_options))
    print(f"Total experiments: {total_configs}")
    print(f"{'='*80}\n")

    experiment_num = 0

    for epochs in epochs_options:
        for batch_size in batch_sizes:
            for opt_name in optimizers_config:
                for lr in learning_rates:
                    for pin_mem in pin_memory_options:
                        experiment_num += 1

                        print(f"\n{'='*80}")
                        print(f"Experiment {experiment_num}/{total_configs}")
                        print(f"Epochs={epochs}, Batch={batch_size}, Optimizer={opt_name}, LR={lr}, pin_memory={pin_mem}")
                        print(f"{'='*80}")

                        train_loader, val_loader, test_loader = prepare_datasets(
                            dataset_name, batch_size, pin_mem
                        )

                        for model_name in models_config:
                            print(f"\n>>> Training {model_name.upper()} <<<")

                            model = get_model(model_name, num_classes=10)

                            if opt_name == 'SGD':
                                optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                            else:
                                optimizer = optim.Adam(model.parameters(), lr=lr)

                            history = train_model(model, train_loader, val_loader,
                                                optimizer, epochs=epochs, use_amp=True)

                            test_acc = test_model(model, test_loader, device)

                            result_row = {
                                'Dataset': dataset_name,
                                'Epochs': epochs,
                                'Batch Size': batch_size,
                                'Optimizer': opt_name,
                                'Learning Rate': lr,
                                'Model': model_name.upper(),
                                'Test Accuracy (%)': round(test_acc, 2),
                                'Best Val Acc (%)': round(max(history['val_accs']), 2),
                                'pin_memory': pin_mem
                            }

                            print(f"\n✓ Test Accuracy: {test_acc:.2f}%")
                            print(f"✓ Best Val Accuracy: {max(history['val_accs']):.2f}%")

                            pd.DataFrame([result_row]).to_csv(csv_filepath, mode='a', header=False, index=False)

                            if test_acc > best_model_info['accuracy']:
                                best_model_info['accuracy'] = test_acc
                                best_model_info['model'] = model
                                best_model_info['config'] = {
                                    'epochs': epochs,
                                    'batch_size': batch_size,
                                    'optimizer': opt_name,
                                    'learning_rate': lr,
                                    'model_name': model_name,
                                    'pin_memory': pin_mem
                                }

                                model_filename = f'best_model_q1a_{dataset_name.lower()}.pth'
                                model_filepath = os.path.join(DRIVE_RESULTS_FOLDER, model_filename)
                                torch.save({
                                    'model_state_dict': best_model_info['model'].state_dict(),
                                    'config': best_model_info['config'],
                                    'test_accuracy': best_model_info['accuracy']
                                }, model_filepath)



    print(f"  Best Config: {best_model_info['config']}")
    print(f"  Test Accuracy: {best_model_info['accuracy']:.2f}%")
    results_df = pd.read_csv(csv_filepath)
    return results_df, best_model_info

print("✓ Q1(a) experiment function defined")

# ============================================================================
# Q1(b) - SVM Data Preparation
# ============================================================================
def prepare_svm_data(dataset_name='MNIST'):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == 'MNIST':
        full_dataset = datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform)
        test_dataset = datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform)
    else:
        full_dataset = datasets.FashionMNIST(root='./data', train=True,
                                             download=True, transform=transform)
        test_dataset = datasets.FashionMNIST(root='./data', train=False,
                                             download=True, transform=transform)
    train_samples,test_samples=10000,5000
    train_subset = torch.utils.data.Subset(full_dataset, range(train_samples))
    test_subset = torch.utils.data.Subset(test_dataset, range(test_samples))

    X_train = torch.stack([x[0] for x in train_subset]).numpy().reshape(train_samples, -1)
    y_train = np.array([x[1] for x in train_subset])

    X_test = torch.stack([x[0] for x in test_subset]).numpy().reshape(test_samples, -1)
    y_test = np.array([x[1] for x in test_subset])

    return X_train, y_train, X_test, y_test
print("✓ SVM data preparation function defined")

# ============================================================================
# Q1(b) - SVM Experiment Function
# ============================================================================
def run_q1b_experiments(dataset_name='MNIST'):

    print(f"\n{'='*80}")
    print(f"Running Q1(b) SVM experiments on {dataset_name}")
    print(f"{'='*80}\n")

    X_train, y_train, X_test, y_test = prepare_svm_data(dataset_name)

    best_svm_info = {'accuracy': 0, 'model': None, 'config': {}}
    kernels = ['poly', 'rbf']

    csv_filepath = os.path.join(DRIVE_RESULTS_FOLDER, f'{dataset_name.lower()}_q1b_svm_results.csv')

    experiment_num = 0
    total_experiments = len(kernels)

    for kernel in kernels:
        experiment_num += 1
        print(f"\n[{experiment_num}/{total_experiments}] Training SVM: kernel={kernel}")

        svm = SVC(kernel=kernel, max_iter=1000, gamma='scale')

        start_time = time.time()
        svm.fit(X_train, y_train)
        train_time = (time.time() - start_time) * 1000

        y_pred = svm.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred) * 100

        result_row = {
            'Dataset': dataset_name,
            'Kernel': kernel,
            'Test Accuracy (%)': round(accuracy, 2),
            'Train Time (ms)': round(train_time, 2)
        }

        pd.DataFrame([result_row]).to_csv(csv_filepath, mode='a', header=False, index=False)

        print(f"✓ Accuracy: {accuracy:.2f}%, Train Time: {train_time:.2f} ms")

        if accuracy > best_svm_info['accuracy']:
            best_svm_info['accuracy'] = accuracy
            best_svm_info['model'] = svm
            best_svm_info['config'] = {
                'kernel': kernel
            }

            model_filename = f'best_svm_q1b_{dataset_name.lower()}.pkl'
            model_filepath = os.path.join(DRIVE_RESULTS_FOLDER, model_filename)
            with open(model_filepath, 'wb') as f:
                pickle.dump({
                    'model': best_svm_info['model'],
                    'config': best_svm_info['config'],
                    'test_accuracy': best_svm_info['accuracy']
                }, f)

    print(f"\n{'='*80}")
    print(f"Q1(b) {dataset_name} SVM experiments completed!")
    print(f"Best Config: {best_svm_info['config']}")
    print(f"Best Test Accuracy: {best_svm_info['accuracy']:.2f}%")
    print(f"{'='*80}")

    results_df = pd.read_csv(csv_filepath)
    return results_df, best_svm_info

print("✓ Q1(b) experiment function defined")

# ============================================================================
# Q2 - CPU vs GPU Experiments Function
# ============================================================================
def run_q2_experiments():
    print(f"\n{'='*80}")
    print(f"Running Q2 experiments: CPU vs GPU on FashionMNIST")
    print(f"Training for 2 epochs")
    print(f"{'='*80}\n")

    best_model_info = {'accuracy': 0, 'model': None, 'config': {}}
    batch_size = 16
    learning_rate = 0.001
    epochs = 2
    optimizers_config = ['Adam','SGD']
    models_config = ['resnet18', 'resnet50']
    devices_test = ['cuda','cpu'] if torch.cuda.is_available() else ['cpu']

    csv_filepath = os.path.join(DRIVE_RESULTS_FOLDER, 'fashionmnist_q2_cpu_gpu_results.csv')

    if not torch.cuda.is_available():
        print("Only CPU experiments will run.")

    experiment_num = 0
    total_experiments = len(devices_test) * len(optimizers_config) * len(models_config)

    for device_name in devices_test:
        device_use = torch.device(device_name)

        for opt_name in optimizers_config:
            print(f"\n{'='*60}")
            print(f"Device: {device_name.upper()}, Optimizer: {opt_name}")
            print(f"{'='*60}\n")

            train_loader, val_loader, test_loader = prepare_datasets(
                'FashionMNIST', batch_size
            )

            for model_name in models_config:
                experiment_num += 1
                print(f"\n[{experiment_num}/{total_experiments}] Training {model_name.upper()} on {device_name.upper()}...")

                model = get_model(model_name, num_classes=10)
                model = model.to(device_use)

                if opt_name == 'SGD':
                    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
                else:
                    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

                print(f"Calculating FLOPs for {model_name}...")
                flops = count_flops(model)

                start_time = time.time()
                history = train_model(model, train_loader, val_loader,
                                    optimizer, epochs=epochs,
                                    device=device_use,
                                    use_amp=(device_name=='cuda'))
                train_time = (time.time() - start_time) * 1000

                test_acc = test_model(model, test_loader, device_use)

                result_row = {
                    'Compute': device_name.upper(),
                    'Batch Size': batch_size,
                    'Optimizer': opt_name,
                    'Learning Rate': learning_rate,
                    'Model': model_name.upper(),
                    'Test Accuracy (%)': round(test_acc, 2),
                    'Train Time (ms)': round(train_time, 2),
                    'FLOPs': f"{flops/1e9:.2f}G"
                }

                pd.DataFrame([result_row]).to_csv(csv_filepath, mode='a', header=False, index=False)

                print(f"\n✓ Test Acc: {test_acc:.2f}%")
                print(f"✓ Train Time: {train_time:.2f} ms")
                print(f"✓ FLOPs: {flops/1e9:.2f}G")

                if test_acc > best_model_info['accuracy']:
                    best_model_info['accuracy'] = test_acc
                    best_model_info['model'] = model
                    best_model_info['config'] = {
                        'device': device_name,
                        'batch_size': batch_size,
                        'optimizer': opt_name,
                        'learning_rate': learning_rate,
                        'model_name': model_name,
                        'epochs': epochs
                    }

                    model_filename = 'best_model_q2_fashionmnist.pth'
                    model_filepath = os.path.join(DRIVE_RESULTS_FOLDER, model_filename)
                    torch.save({
                        'model_state_dict': best_model_info['model'].state_dict(),
                        'config': best_model_info['config'],
                        'test_accuracy': best_model_info['accuracy']
                    }, model_filepath)

    print(f"\n{'='*80}")
    print(f"Q2 experiments completed!")
    print(f"Best Config: {best_model_info['config']}")
    print(f"Best Test Accuracy: {best_model_info['accuracy']:.2f}%")
    print(f"{'='*80}")

    results_df = pd.read_csv(csv_filepath)
    return results_df, best_model_info
print("✓ Q2 experiment function defined")

# ============================================================================
# RUN Q1(a) - MNIST EXPERIMENTS
# ============================================================================
print("\n" + "="*80)
print("STARTING Q1(a) EXPERIMENTS - MNIST")
print("="*80)

mnist_results_q1a, mnist_best_model = run_q1a_experiments('MNIST')

print("\n" + "="*80)
print("MNIST Q1(a) RESULTS SUMMARY")
print("="*80)
print(mnist_results_q1a.to_string(index=False))

# ============================================================================
# RUN Q1(a) - FASHIONMNIST EXPERIMENTS
# ============================================================================
print("\n" + "="*80)
print("STARTING Q1(a) EXPERIMENTS - FASHIONMNIST")
print("="*80)
fashion_results_q1a, fashion_best_model = run_q1a_experiments('FashionMNIST')
print("\n" + "="*80)
print("FASHIONMNIST Q1(a) RESULTS SUMMARY")
print("="*80)
print(fashion_results_q1a.to_string(index=False))

# ============================================================================
# RUN Q1(b) - MNIST SVM EXPERIMENTS
# ============================================================================
print("\n" + "="*80)
print("STARTING Q1(b) SVM EXPERIMENTS - MNIST")
print("="*80)
mnist_svm_results, mnist_best_svm = run_q1b_experiments('MNIST')
print("\n" + "="*80)
print("MNIST SVM RESULTS")
print("="*80)
print(mnist_svm_results.to_string(index=False))

# ============================================================================
# RUN Q1(b) - FASHIONMNIST SVM EXPERIMENTS
# ============================================================================
print("\n" + "="*80)
print("STARTING Q1(b) SVM EXPERIMENTS - FASHIONMNIST")
print("="*80)
fashion_svm_results, fashion_best_svm = run_q1b_experiments('FashionMNIST')
print("\n" + "="*80)
print("FASHIONMNIST SVM RESULTS")
print("="*80)
print(fashion_svm_results.to_string(index=False))

# ============================================================================
# RUN Q2 - CPU vs GPU EXPERIMENTS
# ============================================================================
print("\n" + "="*80)
print("STARTING Q2 EXPERIMENTS - CPU vs GPU")
print("="*80)
q2_results, q2_best_model = run_q2_experiments()
print("\n" + "="*80)
print("Q2 RESULTS - CPU vs GPU COMPARISON")
print("="*80)
print(q2_results.to_string(index=False))